# Google Colab Pro - LAMMPS + Quantum ESPRESSO (GPU) 構築ノートブック
このノートブックは、Colab環境上で初回のみQEとLAMMPSをコンパイルし、以降はGoogle Driveに保存したバイナリを再利用して素早く計算を開始するためのものです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Miniconda と Python パッケージのインストール

In [ ]:
%%bash
MINICONDA_INSTALLER_SCRIPT=Miniconda3-latest-Linux-x86_64.sh
MINICONDA_PREFIX=/usr/local
wget -q https://repo.continuum.io/miniconda/$MINICONDA_INSTALLER_SCRIPT
chmod +x $MINICONDA_INSTALLER_SCRIPT
./$MINICONDA_INSTALLER_SCRIPT -b -f -p $MINICONDA_PREFIX
conda install -y -c conda-forge numpy
pip install ase pymdi

## 2. NVIDIA HPC SDK のインストール (GPUコンパイル用)
初回はダウンロードに5〜10分程度かかります。

In [ ]:
%%bash
if [ ! -f /content/drive/MyDrive/mdi_binaries/pw.x ]; then
  echo "Installing NVIDIA HPC SDK..."
  echo "deb [trusted=yes] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /" | sudo tee /etc/apt/sources.list.d/nvhpc.list
  sudo apt-get update -y
  sudo apt-get install -y nvhpc-23-9
else
  echo "pw.x is already built in Google Drive. Skipping NVHPC SDK installation."
fi

## 3. Quantum ESPRESSO と LAMMPS のビルド

In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/mdi_binaries

# Quantum ESPRESSO GPUビルド
if [ ! -f /content/drive/MyDrive/mdi_binaries/pw.x ]; then
  export NVHPC_HOME=/opt/nvidia/hpc_sdk/Linux_x86_64/23.9
  export PATH=$NVHPC_HOME/compilers/bin:$PATH
  export LD_LIBRARY_PATH=$NVHPC_HOME/compilers/lib:$LD_LIBRARY_PATH

  echo "Compiling Quantum ESPRESSO with GPU..."
  git clone https://gitlab.com/QEF/q-e.git
  cd q-e
  git checkout qe-7.2
  ./configure --with-cuda="yes" --with-cuda-cc=70,75,80  # V100, T4, A100対応
  make pw -j$(nproc)
  cp bin/pw.x /content/drive/MyDrive/mdi_binaries/
  echo "QE Compilation done!"
  cd ..
fi

# LAMMPS MDIビルド
if [ ! -f /content/drive/MyDrive/mdi_binaries/lmp ]; then
  echo "Compiling LAMMPS with MDI..."
  git clone -b stable_2Aug2023_update3 https://github.com/lammps/lammps.git
  mkdir -p lammps/build && cd lammps/build
  cmake ../cmake -D PKG_MDI=yes -D MDI_USE_PYTHON_PLUGINS=OFF
  make -j$(nproc)
  cp lmp /content/drive/MyDrive/mdi_binaries/
  echo "LAMMPS Compilation done!"
  cd ../..
fi

echo "All binaries are safely stored in /content/drive/MyDrive/mdi_binaries/"


## 4. プログラムの実行パス設定と計算の開始
Google Driveに保存されたバイナリを `/content/bin/` にコピーして実行可能な状態にします。

In [ ]:
%%bash
mkdir -p /content/bin
cp /content/drive/MyDrive/mdi_binaries/pw.x /content/bin/ 2>/dev/null || true
cp /content/drive/MyDrive/mdi_binaries/lmp /content/bin/ 2>/dev/null || true
chmod +x /content/bin/pw.x /content/bin/lmp

# パスの確認
ls -l /content/bin/

# 準備ができたら、以下のようにtar.gzを解凍して run_colab.sh を実行します。
# tar -xzvf /content/drive/MyDrive/colab_transfer.tar.gz -C /content/
# cd /content/SiC_3x3x10_MD
# nohup bash run_colab.sh > nohup.out 2>&1 &
# tail -f nohup.out